# Laya V2.5 — Sliding CeNN + Multiscale Recurrent CeNN
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_Integrated_Memory_V25_WindowDelta_CeNN_Decision_Colab.ipynb)

Sliding attention is replaced by **bidirectional sliding CeNN**; global attention by **multiscale recurrent CeNN**. The final decision head uses pooled context and a shared option MLP. FFNs, norms, residuals, data splits and training stages remain intact. The filename is retained for existing links.

This revision removes Delta and FLA from the trained/exported model. Local transfer keeps the successful **0.01 → 0.002** schedule. Joint/gold training now uses smaller rates and a short warmup after the saved run failed at Stage B step 74. Each phase resets AMP scaling and can restore the best dev checkpoint and reduce its learning rates on numerical failure. Accuracy and speed still require a completed GPU evaluation.

The previous Delta run reached 55.6% teacher agreement at the end of recovery; its partial gold run has no final test result. Its outputs are preserved in `notebooks/archive/Laya_V25_Delta_recovery_20261006.ipynb`. Use a fresh runtime and output directory: Delta checkpoints cannot initialize the sliding CeNN architecture.

The latest sliding CeNN run completed Stage A; layer 25 reached cosine **0.9531** and NMSE **0.0960**. Its recovery failure is archived at `notebooks/archive/Laya_V25_SlidingCeNN_recovery_failure_20261006.ipynb`. Existing sliding CeNN Stage A checkpoints remain compatible: keep `/content/Laya_V25_SlidingCeNN/stage_A` and leave `RESUME_STAGE_A=True`.


In [1]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
#@title 2. Imports, configuration, source teacher/student
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import (
    layer_kinds, replacement_for, core_parameters as v25_core_parameters,
    SlidingCeNN, MultiscaleCeNN, LinearMaskEncoder,
    PooledMarkerDecisionModel, attention_audit, export_gate, save_v25, load_v25,
)

from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)

SEED = 2026
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}

# Stages, training counts and data splits are retained.
LOCAL_STEPS = 2000 #@param {type:"integer"}
GLOBAL_STEPS = 3200 #@param {type:"integer"}
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 100 #@param {type:"integer"}
TEST_CASES = 400 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}

# Keep the successful local fit. Joint training updates all layers together;
# use lower rates and short warmup after the observed Stage B divergence.
LOCAL_LR = 0.01 #@param {type:"number"}
LOCAL_LR_MIN = 0.002 #@param {type:"number"}
GLOBAL_CORE_LR = 0.001 #@param {type:"number"}
GLOBAL_CORE_LR_MIN = 0.0002 #@param {type:"number"}
GLOBAL_HEAD_LR = 0.001 #@param {type:"number"}
GLOBAL_HEAD_LR_MIN = 2e-05 #@param {type:"number"}
DIRECT_FULL_STEPS = 900 #@param {type:"integer"}
DIRECT_FULL_CORE_LR = 0.003 #@param {type:"number"}
DIRECT_FULL_CORE_LR_MIN = 6e-05 #@param {type:"number"}
DIRECT_FULL_HEAD_LR = 5e-05 #@param {type:"number"}
DIRECT_FULL_HEAD_LR_MIN = 1e-05 #@param {type:"number"}
DIRECT_COMPACT_STEPS = 700 #@param {type:"integer"}
DIRECT_COMPACT_CORE_LR = 0.001 #@param {type:"number"}
DIRECT_COMPACT_CORE_LR_MIN = 6e-05 #@param {type:"number"}
DIRECT_COMPACT_HEAD_LR = 0.0001 #@param {type:"number"}
DIRECT_COMPACT_HEAD_LR_MIN = 2e-05 #@param {type:"number"}
COMPACT_HEAD_LR = 0.001 #@param {type:"number"}
COMPACT_HEAD_LR_MIN = 0.0002 #@param {type:"number"}
JOINT_CORE_LR = 0.003 #@param {type:"number"}
JOINT_CORE_LR_MIN = 6e-05 #@param {type:"number"}
JOINT_HEAD_LR = 0.001 #@param {type:"number"}
JOINT_HEAD_LR_MIN = 2e-05 #@param {type:"number"}
RUN_COMPACT_HEAD = True #@param {type:"boolean"}
COMPACT_DIM = 384 #@param {type:"integer"}
COMPACT_LAYERS = 2 #@param {type:"integer"}
COMPACT_STEPS = 700 #@param {type:"integer"}
COMPACT_JOINT_STEPS = 1200 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V25_SlidingCeNN")

# -------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required for full training and latency evaluation."
    )

device = torch.device("cuda")

amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported(including_emulation=False)
    else torch.float16
)

# -------------------------------------------------------------------
# Load teacher and student
# -------------------------------------------------------------------

source_dir = Path(
    snapshot_download(
        SOURCE_MODEL,
        allow_patterns=[
            "rl_agent_config.json",
            "model.safetensors",
            "encoder/*",
            "tokenizer/*",
        ],
    )
)

teacher, source_cfg = build_checkpoint_model(source_dir)
student_full, _ = build_checkpoint_model(source_dir)

tokenizer = AutoTokenizer.from_pretrained(
    str(source_dir / "tokenizer")
)

teacher.to(device).eval().requires_grad_(False)
student_full.to(device).eval().requires_grad_(False)

# Disable auto-compilation for consistent teacher/student benchmarking.
teacher.encoder.config.reference_compile = False
student_full.encoder.config.reference_compile = False
kinds = layer_kinds(student_full.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
replacement_layers = list(range(len(kinds)))
ARCHITECTURE = {
    "window": 128, "cell_dim": 256, "cell_steps": 2,
    "scales": [1, 4, 16], "local_mixer": "sliding_cenn",
    "sliding_cell_dim": 384, "sliding_steps": 2,
}
MAX_ACCURACY_DROP = 0.02 #@param {type:"number"}
MIN_SPEEDUP = 1.10 #@param {type:"number"}
PUSH_TO_HUB = True #@param {type:"boolean"}
HF_REPO_ID = "" #@param {type:"string"}
HF_PRIVATE = True #@param {type:"boolean"}
print("V2.5 layer map:", dict(enumerate(kinds)))
print("Architecture:", ARCHITECTURE)

print("device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "| amp:", amp_dtype)
print("source:", SOURCE_MODEL)
print(
    "full-attention layers:",
    full_layers,
    "| count:",
    len(full_layers),
)

print("\nTuned V2.5 configuration")
print("  LOCAL_STEPS       :", LOCAL_STEPS)
print("  GLOBAL_STEPS      :", GLOBAL_STEPS)
print("  TRAIN / DEV / TEST:", TRAIN_CASES, "/", DEV_CASES, "/", TEST_CASES)
print("  LOCAL_LR          :", LOCAL_LR, "->", LOCAL_LR_MIN)
print("  GLOBAL_CORE_LR   :", GLOBAL_CORE_LR, "->", GLOBAL_CORE_LR_MIN)
print("  GLOBAL_HEAD_LR    :", GLOBAL_HEAD_LR, "->", GLOBAL_HEAD_LR_MIN)
print("  DIRECT_FULL_STEPS :", DIRECT_FULL_STEPS)
print("  DIRECT_COMPACT    :", DIRECT_COMPACT_STEPS)
print("  COMPACT_DIM       :", COMPACT_DIM)
print("  COMPACT_STEPS     :", COMPACT_STEPS)
print("  COMPACT_JOINT     :", COMPACT_JOINT_STEPS)

assert full_layers
from tinycenn_lm.decision_training import (
    require_finite, guarded_step, gold_checkpoint_key, trainable_snapshot,
    save_stage, load_stage, forward_with_markers, marker_alignment_loss,
    run_recovery_phase, decision_distill_loss,
)
# New output directory and architecture metadata prevent reuse of Delta checkpoints.
RESUME_STAGE_A = True #@param {type:"boolean"}
# Change OUTPUT_DIR to a persistent mounted directory to retain progress across sessions.
STAGE_A_DIR = OUTPUT_DIR / "stage_A"
scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.0)
print("Recovery LR core/head:", GLOBAL_CORE_LR, GLOBAL_HEAD_LR)
MARKER_ALIGNMENT_WEIGHT = 0.05 #@param {type:"number"}

RECOVERY_WARMUP = 20 #@param {type:"integer"}
RECOVERY_MAX_RESTARTS = 3 #@param {type:"integer"}
RECOVERY_BACKOFF = 0.25 #@param {type:"number"}
recovery_reports = {}


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

V2.5 layer map: {0: 'full_attention', 1: 'sliding_attention', 2: 'sliding_attention', 3: 'full_attention', 4: 'sliding_attention', 5: 'sliding_attention', 6: 'full_attention', 7: 'sliding_attention', 8: 'sliding_attention', 9: 'full_attention', 10: 'sliding_attention', 11: 'sliding_attention', 12: 'full_attention', 13: 'sliding_attention', 14: 'sliding_attention', 15: 'full_attention', 16: 'sliding_attention', 17: 'sliding_attention', 18: 'full_attention', 19: 'sliding_attention', 20: 'sliding_attention', 21: 'full_attention', 22: 'sliding_attention', 23: 'sliding_attention', 24: 'full_attention', 25: 'sliding_attention', 26: 'sliding_attention', 27: 'full_attention'}
Architecture: {'window': 128, 'cell_dim': 256, 'cell_steps': 2, 'scales': [1, 4, 16], 'local_mixer': 'sliding_cenn', 'sliding_cell_dim': 384, 'sliding_steps': 2}
device: Tesla T4
torch: 2.11.0+cu130 | amp: torch.float16
source: convaiinnovations/laya-typed-decisions
full-attention layers: [0, 3, 6, 9, 12, 15, 18, 21, 24, 

In [6]:
#@title 3. Sliding/global CeNN preflight on the actual GPU and dtype
def v25_indices(model):
    return [i for i, layer in enumerate(model.encoder.layers)
            if isinstance(layer.attn, (SlidingCeNN, MultiscaleCeNN))]

# Both replacement types must have finite outputs/gradients in the actual AMP dtype.
# Compare to FP32 and verify that extra padding cannot change valid outputs.
for kind in ("sliding_attention", "full_attention"):
    index = kinds.index(kind)
    probe = replacement_for(student_full.encoder.layers[index].attn, kind, ARCHITECTURE).to(device).eval()
    x = torch.randn(2, 37, student_full.encoder.config.hidden_size, device=device)
    mask = torch.ones(2, 37, dtype=torch.long, device=device)
    mask[1, -7:] = 0
    with torch.no_grad():
        reference = probe(x, attention_mask=mask)[0]
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        actual = probe(x, attention_mask=mask)[0]
        padded = probe(F.pad(x, (0, 0, 0, 11)), attention_mask=F.pad(mask, (0, 11)))[0]
    require_finite(actual, kind + " preflight")
    torch.testing.assert_close(actual.float(), reference.float(), rtol=0.04, atol=0.03)
    torch.testing.assert_close(actual, padded[:, :37], rtol=0.01, atol=0.01)
    assert not padded[:, 37:].count_nonzero()
    grads = torch.autograd.grad(actual.float().square().mean(), probe.trainable_core_parameters())
    assert all(torch.isfinite(g).all() for g in grads)
    assert sum(g.abs().sum() for g in grads) > 0
    print(kind, "CeNN forward/backward preflight passed:", torch.cuda.get_device_name(), amp_dtype)
    del probe, x, mask, reference, actual, padded, grads
torch.cuda.empty_cache()


sliding_attention CeNN forward/backward preflight passed: Tesla T4 torch.float16
full_attention CeNN forward/backward preflight passed: Tesla T4 torch.float16


In [7]:
#@title 4. Build leak-free train / dev / untouched official test
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    """Align the benchmark gold distribution to the exact marker order used by build_sequence."""
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        # Score markers are generated in rubric order. The benchmark normally
        # uses numeric string keys ("0", "1", ...); sort those numerically.
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match the encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not present in option labels {labels!r}")
    gold_index = canon_labels.index(gold_label)
    return labels, gold_probs.tolist(), int(gold_index)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "), add_special_tokens=False
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer, state, q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue

            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: marker/gold option mismatch "
                    f"({len(markers)} markers vs {len(labels)} labels)"
                )

            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": gold_index,
            })

    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

# Split only the official TRAIN rows into fit/dev. Never tune on official TEST.
all_train_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
split_rng = random.Random(SEED)
split_rng.shuffle(all_train_rows)
assert TRAIN_CASES + DEV_CASES <= len(all_train_rows)

train_rows = all_train_rows[:TRAIN_CASES]
dev_rows = all_train_rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, shuffle_items=True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, shuffle_items=True)

print(
    "cases  train/dev:",
    len(train_rows), "/", len(dev_rows),
)
print(
    "decisions train/dev:",
    len(train_items), "/", len(dev_items),
)
print("official test: deferred until the final evaluation cell")
assert len(train_items) >= 64 and len(dev_items) >= 32

def batch(items, offset, n=BATCH):
    selected = [items[(offset+i) % len(items)] for i in range(n)]
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove(); h2.remove()
    return got

def local_loss(pred, target, valid):
    m = valid[:, :, None].float()
    mse = (((pred.float()-target.float())**2)*m).sum() / (m.sum()*pred.shape[-1]).clamp_min(1)
    power = ((target.float()**2)*m).sum() / (m.sum()*target.shape[-1]).clamp_min(1)
    nmse = mse / power.clamp_min(1e-8)
    cos = F.cosine_similarity(pred.float()[valid], target.float()[valid], dim=-1).mean()
    return nmse + 1.10*(1-cos), nmse, cos


cases  train/dev: 600 / 100
decisions train/dev: 3000 / 500
official test: deferred until the final evaluation cell


In [8]:
#@title 5. Stage A — progressive local transfer
def cosine_lr(step, total_steps, start_lr, end_lr):
    """Exact cosine decay: step 0=start_lr, last step=end_lr."""
    if total_steps <= 1:
        return float(end_lr)
    x = min(1.0, max(0.0, step / float(total_steps - 1)))
    return float(end_lr + 0.5 * (start_lr - end_lr) * (1.0 + math.cos(math.pi * x)))

stage_a_metadata = {"protocol": "sliding-cenn-1", "source_model": SOURCE_MODEL,
                  "source_revision": source_dir.name, "seed": SEED, "local_steps": LOCAL_STEPS,
                  "local_lr": [LOCAL_LR, LOCAL_LR_MIN], "max_len": MAX_LEN,
                  "train_cases": TRAIN_CASES, "dev_cases": DEV_CASES, "batch": BATCH,
                  "architecture": ARCHITECTURE}
if RESUME_STAGE_A and (STAGE_A_DIR / "model.safetensors").exists():
    for layer_idx in replacement_layers:
        layer = student_full.encoder.layers[layer_idx]
        if not isinstance(layer.attn, (SlidingCeNN, MultiscaleCeNN)):
            layer.attn = replacement_for(layer.attn, kinds[layer_idx], ARCHITECTURE).to(device)
    student_full.encoder = LinearMaskEncoder(student_full.encoder)
    load_stage(student_full, STAGE_A_DIR, stage_a_metadata)
    student_full.eval().requires_grad_(False)
    print("Resumed completed Stage A:", STAGE_A_DIR)
else:
    local_report = {}

    for layer_idx in replacement_layers:
        print(f"\n=== {kinds[layer_idx]} layer {layer_idx} ===")
        original = student_full.encoder.layers[layer_idx].attn
        repl = replacement_for(original, kinds[layer_idx], ARCHITECTURE).to(device)
        params = repl.trainable_core_parameters()
        opt = torch.optim.AdamW(params, lr=LOCAL_LR, betas=(0.9, 0.95), weight_decay=1e-4)
        best = None; best_score = float("inf")

        probe_b = batch(dev_items, layer_idx*11, min(5, BATCH+1))
        probe = attn_io(student_full, layer_idx, probe_b)
        probe_valid = probe_b["attention_mask"].bool()

        for step in range(LOCAL_STEPS):
            local_lr = cosine_lr(step, LOCAL_STEPS, LOCAL_LR, LOCAL_LR_MIN)
            opt.param_groups[0]["lr"] = local_lr
            b = batch(train_items, step*BATCH + layer_idx*17)
            target = attn_io(student_full, layer_idx, b)
            valid = b["attention_mask"].bool()

            repl.train(); repl.out_drop.eval(); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    target["x"],
                    position_embeddings=target["pos"],
                    attention_mask=target["mask"],
                )
                loss, nmse, cos = local_loss(pred, target["y"], valid)
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite local loss at layer {layer_idx}")
            guarded_step(loss, opt, scaler, [(params, 1.0)], context=f"stage 5 step {step+1}")

            if step == 0 or (step+1) % max(20, LOCAL_STEPS//4) == 0 or step+1 == LOCAL_STEPS:
                repl.eval()
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
                    pp, _ = repl(
                        probe["x"],
                        position_embeddings=probe["pos"],
                        attention_mask=probe["mask"],
                    )
                    ploss, pnmse, pcos = local_loss(pp, probe["y"], probe_valid)
                score = float(ploss)
                if score < best_score:
                    best_score = score
                    best = {k:v.detach().cpu().clone() for k,v in repl.state_dict().items()}
                print(
                    f"step {step+1:03d}/{LOCAL_STEPS} "
                    f"nmse={float(pnmse):.4f} cos={float(pcos):.4f} lr={local_lr:.6f}"
                )

        if best is not None:
            repl.load_state_dict(best)
        repl.eval()
        student_full.encoder.layers[layer_idx].attn = repl
        local_report[str(layer_idx)] = {"best_local_score": best_score}
        torch.cuda.empty_cache()

    print("\nconverted:", v25_indices(student_full))
    assert v25_indices(student_full) == replacement_layers
    student_full.encoder = LinearMaskEncoder(student_full.encoder)
    print("All encoder attention replaced; FFNs/norms/residuals retained.")
    print("V2.5 new core params:", f"{sum(p.numel() for p in v25_core_parameters(student_full)):,}")

    save_stage(student_full, STAGE_A_DIR, stage_a_metadata)
    print("Saved completed Stage A:", STAGE_A_DIR)



=== full_attention layer 0 ===
step 001/2000 nmse=1.9504 cos=0.1971 lr=0.010000
step 500/2000 nmse=0.2312 cos=0.8611 lr=0.008832
step 1000/2000 nmse=0.2032 cos=0.8783 lr=0.006003
step 1500/2000 nmse=0.1816 cos=0.8875 lr=0.003173
step 2000/2000 nmse=0.1774 cos=0.8899 lr=0.002000

=== sliding_attention layer 1 ===
step 001/2000 nmse=5.4825 cos=0.1769 lr=0.010000
step 500/2000 nmse=0.1255 cos=0.9267 lr=0.008832
step 1000/2000 nmse=0.1029 cos=0.9410 lr=0.006003
step 1500/2000 nmse=0.1017 cos=0.9419 lr=0.003173
step 2000/2000 nmse=0.0959 cos=0.9456 lr=0.002000

=== sliding_attention layer 2 ===
step 001/2000 nmse=6.7875 cos=0.2536 lr=0.010000
step 500/2000 nmse=0.1659 cos=0.9046 lr=0.008832
step 1000/2000 nmse=0.1430 cos=0.9184 lr=0.006003
step 1500/2000 nmse=0.1312 cos=0.9256 lr=0.003173
step 2000/2000 nmse=0.1288 cos=0.9272 lr=0.002000

=== full_attention layer 3 ===
step 001/2000 nmse=3.8689 cos=0.1604 lr=0.010000
step 500/2000 nmse=0.1931 cos=0.8898 lr=0.008832
step 1000/2000 nmse=0.15

/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: stage 5 step 3: skipped overflowing gradient update; scale 4096 -> 2048
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")


step 500/2000 nmse=0.1006 cos=0.9384 lr=0.008832
step 1000/2000 nmse=0.0850 cos=0.9467 lr=0.006003
step 1500/2000 nmse=0.0754 cos=0.9536 lr=0.003173
step 2000/2000 nmse=0.0726 cos=0.9542 lr=0.002000

=== full_attention layer 6 ===
step 001/2000 nmse=4.0498 cos=0.1883 lr=0.010000
step 500/2000 nmse=0.2366 cos=0.8618 lr=0.008832
step 1000/2000 nmse=0.2102 cos=0.8773 lr=0.006003
step 1500/2000 nmse=0.1877 cos=0.8903 lr=0.003173
step 2000/2000 nmse=0.1832 cos=0.8927 lr=0.002000

=== sliding_attention layer 7 ===
step 001/2000 nmse=10.3137 cos=0.2331 lr=0.010000


/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: stage 5 step 2: skipped overflowing gradient update; scale 4096 -> 2048
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")


step 500/2000 nmse=0.1282 cos=0.9286 lr=0.008832
step 1000/2000 nmse=0.1101 cos=0.9375 lr=0.006003
step 1500/2000 nmse=0.0941 cos=0.9474 lr=0.003173
step 2000/2000 nmse=0.0894 cos=0.9497 lr=0.002000

=== sliding_attention layer 8 ===
step 001/2000 nmse=5.4794 cos=0.2189 lr=0.010000
step 500/2000 nmse=0.1800 cos=0.8819 lr=0.008832
step 1000/2000 nmse=0.1629 cos=0.8951 lr=0.006003
step 1500/2000 nmse=0.1497 cos=0.9028 lr=0.003173
step 2000/2000 nmse=0.1458 cos=0.9057 lr=0.002000

=== full_attention layer 9 ===
step 001/2000 nmse=3.0932 cos=0.2106 lr=0.010000
step 500/2000 nmse=0.3361 cos=0.8178 lr=0.008832
step 1000/2000 nmse=0.3024 cos=0.8381 lr=0.006003
step 1500/2000 nmse=0.2933 cos=0.8427 lr=0.003173
step 2000/2000 nmse=0.2846 cos=0.8481 lr=0.002000

=== sliding_attention layer 10 ===
step 001/2000 nmse=3.3846 cos=0.2016 lr=0.010000


...

step 500/2000 nmse=0.1927 cos=0.8939 lr=0.008832
step 1000/2000 nmse=0.1626 cos=0.9109 lr=0.006003
step 1500/2000 nmse=0.1513 cos=0.9170 lr=0.003173
step 2000/2000 nmse=0.1456 cos=0.9200 lr=0.002000

=== sliding_attention layer 17 ===
step 001/2000 nmse=1.5822 cos=0.4636 lr=0.010000
step 500/2000 nmse=0.1857 cos=0.9021 lr=0.008832
step 1000/2000 nmse=0.1602 cos=0.9155 lr=0.006003
step 1500/2000 nmse=0.1497 cos=0.9215 lr=0.003173
step 2000/2000 nmse=0.1443 cos=0.9243 lr=0.002000

=== full_attention layer 18 ===
step 001/2000 nmse=1.0399 cos=0.2926 lr=0.010000
step 500/2000 nmse=0.2919 cos=0.8396 lr=0.008832
step 1000/2000 nmse=0.2568 cos=0.8580 lr=0.006003
step 1500/2000 nmse=0.2403 cos=0.8710 lr=0.003173
step 2000/2000 nmse=0.2305 cos=0.8754 lr=0.002000

=== sliding_attention layer 19 ===
step 001/2000 nmse=1.6596 cos=0.1707 lr=0.010000


/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: stage 5 step 1: skipped overflowing gradient update; scale 65536 -> 32768
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")
/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: stage 5 step 3: skipped overflowing gradient update; scale 32768 -> 16384
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")
/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: stage 5 step 6: skipped overflowing gradient update; scale 16384 -> 8192
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")


step 500/2000 nmse=0.1935 cos=0.9001 lr=0.008832
step 1000/2000 nmse=0.1519 cos=0.9209 lr=0.006003
step 1500/2000 nmse=0.1389 cos=0.9279 lr=0.003173
step 2000/2000 nmse=0.1290 cos=0.9328 lr=0.002000

=== sliding_attention layer 20 ===
step 001/2000 nmse=1.6317 cos=0.4324 lr=0.010000
step 500/2000 nmse=0.1506 cos=0.9148 lr=0.008832
step 1000/2000 nmse=0.1094 cos=0.9339 lr=0.006003
step 1500/2000 nmse=0.1013 cos=0.9383 lr=0.003173
step 2000/2000 nmse=0.0930 cos=0.9426 lr=0.002000

=== full_attention layer 21 ===
step 001/2000 nmse=0.8234 cos=0.4791 lr=0.010000
step 500/2000 nmse=0.1894 cos=0.9004 lr=0.008832
step 1000/2000 nmse=0.1695 cos=0.9118 lr=0.006003
step 1500/2000 nmse=0.1600 cos=0.9176 lr=0.003173
step 2000/2000 nmse=0.1545 cos=0.9204 lr=0.002000

=== sliding_attention layer 22 ===
step 001/2000 nmse=1.1069 cos=0.5140 lr=0.010000
step 500/2000 nmse=0.1301 cos=0.9282 lr=0.008832
step 1000/2000 nmse=0.1081 cos=0.9396 lr=0.006003
step 1500/2000 nmse=0.0994 cos=0.9439 lr=0.003173
st

/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: stage 5 step 2: skipped overflowing gradient update; scale 65536 -> 32768
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")


step 500/2000 nmse=0.1190 cos=0.9459 lr=0.008832
step 1000/2000 nmse=0.0992 cos=0.9554 lr=0.006003
step 1500/2000 nmse=0.0917 cos=0.9587 lr=0.003173
step 2000/2000 nmse=0.0874 cos=0.9608 lr=0.002000

=== full_attention layer 27 ===
step 001/2000 nmse=0.7399 cos=0.5284 lr=0.010000
step 500/2000 nmse=0.0928 cos=0.9537 lr=0.008832
step 1000/2000 nmse=0.0785 cos=0.9610 lr=0.006003
step 1500/2000 nmse=0.0708 cos=0.9648 lr=0.003173
step 2000/2000 nmse=0.0682 cos=0.9663 lr=0.002000

converted: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
All encoder attention replaced; FFNs/norms/residuals retained.
V2.5 new core params: 21,017,146
Saved completed Stage A: /content/Laya_V25_SlidingCeNN/stage_A


In [9]:
#@title 6. Stage B — end-to-end recovery with original decision head
for p in student_full.parameters():
    p.requires_grad = False

# Core discovery must be independent of requires_grad; this assertion catches regressions.
core_params = v25_core_parameters(student_full)
assert len(core_params) > 0, "CeNN core parameter list is empty; Stage B would not train the replacement."
for p in core_params:
    p.requires_grad = True
print("Global CeNN trainable params:", f"{sum(p.numel() for p in core_params):,}")

head_params = []
for module in (student_full.head, student_full.type_emb, student_full.scorer, student_full.act_head):
    if module is not None:
        for p in module.parameters():
            p.requires_grad = True
            head_params.append(p)

T_DISTILL = 1.25
RANK_MARGIN = 0.15
RANK_WEIGHT = 0.15

def distill_loss(slog, tlog, mask, sact, tact):
    return decision_distill_loss(slog, tlog, mask, sact, tact,
                                 T_DISTILL, RANK_MARGIN, RANK_WEIGHT)

@torch.no_grad()
def validate_pair(student, items=dev_items, limit=128):
    teacher.eval(); student.eval()
    total = agree = 0; js = []
    for off in range(0, min(limit, len(items)), BATCH):
        n = min(BATCH, min(limit, len(items))-off)
        b = batch(items, off, n)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b); sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "student validation logits")
        m = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~m, -1e9)
        sm = sl.float().masked_fill(~m, -1e9)
        tp = torch.softmax(tm, -1); sp = torch.softmax(sm, -1)
        agree += int((tm.argmax(-1) == sm.argmax(-1)).sum())
        total += int(tm.shape[0])
        mid = 0.5*(tp+sp)
        j = 0.5*(tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        j += 0.5*(sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(j.cpu().tolist())
    return {"agreement": agree/max(1,total), "mean_js": float(np.mean(js))}

def agreement_score(metrics):
    return metrics["mean_js"] + 0.50*(1.0-metrics["agreement"])

def recover(name, model, groups, rates, steps, loss_fn, evaluate_fn, score_fn, eval_every=50):
    optimizer = torch.optim.AdamW([
        {"params": params, "lr": rate[0], "weight_decay": 1e-4}
        for (params, _), rate in zip(groups, rates)
    ], betas=(0.9, 0.95))
    def report(row):
        print(f"{name} step={row['step']:04d}/{steps} loss={row['loss']:.4f} "
              f"lr={row['lr']} dev={row['metrics']}")
    result = run_recovery_phase(
        model, optimizer, groups, steps, rates, loss_fn, evaluate_fn, score_fn, name,
        eval_every=eval_every, amp_enabled=amp_dtype == torch.float16,
        warmup_steps=RECOVERY_WARMUP, max_restarts=RECOVERY_MAX_RESTARTS,
        backoff=RECOVERY_BACKOFF, report_fn=report,
    )
    recovery_reports[name] = result
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "recovery_progress.json").write_text(json.dumps(recovery_reports, indent=2, allow_nan=False))
    print(name, "selected dev:", result["metrics"], "restarts:", result["restarts"])
    return result["metrics"]

def full_distillation_step(step):
    b = batch(train_items, step*BATCH + 31)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        tl, ta, teacher_features = forward_with_markers(teacher, b)
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        sl, sa, student_features = forward_with_markers(student_full, b)
        loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)
        return loss + MARKER_ALIGNMENT_WEIGHT * marker_alignment_loss(
            student_features, teacher_features, b["marker_mask"].bool())

v25_full_quality = recover(
    "full-distillation", student_full, [(core_params, 1.0), (head_params, 0.5)],
    [(GLOBAL_CORE_LR, GLOBAL_CORE_LR_MIN), (GLOBAL_HEAD_LR, GLOBAL_HEAD_LR_MIN)],
    GLOBAL_STEPS, full_distillation_step,
    lambda: validate_pair(student_full, items=dev_items, limit=len(dev_items)),
    agreement_score, eval_every=50,
)
student_full.eval().requires_grad_(False)
print("V2.5 full-head quality:", v25_full_quality)
torch.cuda.empty_cache()


Global CeNN trainable params: 21,017,146
full-distillation step=0001/3200 loss=0.7918 lr=[5e-05, 5e-05] dev={'agreement': 0.516, 'mean_js': 0.08075470844854135}
full-distillation step=0050/3200 loss=0.7921 lr=[0.0009998357463776148, 0.000999798789312578] dev={'agreement': 0.456, 'mean_js': 0.1024712317463709}
full-distillation step=0100/3200 loss=0.9675 lr=[0.0009987816213738708, 0.0009985074861829917] dev={'agreement': 0.418, 'mean_js': 0.10202146387984976}
full-distillation step=0150/3200 loss=1.0338 lr=[0.0009967540647038134, 0.0009960237292621712] dev={'agreement': 0.464, 'mean_js': 0.10183650295529514}
full-distillation step=0200/3200 loss=0.9002 lr=[0.0009937580256624, 0.0009923535814364399] dev={'agreement': 0.468, 'mean_js': 0.11420726859587012}
full-distillation step=0250/3200 loss=0.8874 lr=[0.0009898008176239693, 0.0009875060015893623] dev={'agreement': 0.414, 'mean_js': 0.10036006664589513}
full-distillation step=0300/3200 loss=0.9704 lr=[0.0009848921001901888, 0.0009814928

/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: full-distillation step 602: skipped overflowing gradient update; scale 128 -> 64
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")
/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: full-distillation step 602: skipped overflowing gradient update; scale 64 -> 32
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")
/usr/local/lib/python3.13/dist-packages/tinycenn_lm/decision_training.py:66: UserWarning: full-distillation step 602: skipped overflowing gradient update; scale 32 -> 16
  warnings.warn(f"{context}: skipped overflowing gradient update; scale {old_scale:g} -> {scaler.get_scale():g}")


full-distillation step=0650/3200 loss=0.3746 lr=[0.0009251794422012103, 0.0009083448166964825] dev={'agreement': 0.482, 'mean_js': 0.10643810197059066}
full-distillation step=0700/3200 loss=0.7313 lr=[0.0009132775794950339, 0.0008937650348814166] dev={'agreement': 0.436, 'mean_js': 0.09234048212354537}
full-distillation step=0750/3200 loss=0.7351 lr=[0.0009006110017170447, 0.0008782484771033799] dev={'agreement': 0.488, 'mean_js': 0.09364617297030053}
full-distillation step=0800/3200 loss=0.4223 lr=[0.0008872106281655559, 0.0008618330195028058] dev={'agreement': 0.454, 'mean_js': 0.10515879732606118}
full-distillation step=0850/3200 loss=1.0714 lr=[0.0008731091693449036, 0.000844558732447507] dev={'agreement': 0.466, 'mean_js': 0.0928574642566964}
full-distillation step=0900/3200 loss=0.4246 lr=[0.0008583410471186387, 0.0008264677827203325] dev={'agreement': 0.502, 'mean_js': 0.0892631324804388}
full-distillation step=0950/3200 loss=0.8786 lr=[0.0008429423106852609, 0.00080760433058944

In [10]:
#@title 7. Stage C — direct supervised training on benchmark gold data
# This stage is different from distillation: teacher outputs are NOT used in the loss.
# Targets come directly from train_items: gold label + full soft gold distribution.

def _gold_target_tensors(selected, marker_mask, device):
    max_k = marker_mask.shape[1]
    soft = torch.zeros((len(selected), max_k), dtype=torch.float32, device=device)
    hard = torch.zeros(len(selected), dtype=torch.long, device=device)
    for i, item in enumerate(selected):
        g = torch.tensor(item["gold_probs"], dtype=torch.float32, device=device)
        soft[i, :g.numel()] = g
        hard[i] = int(item["gold_index"])
    soft = soft * marker_mask.float()
    soft = soft / soft.sum(-1, keepdim=True).clamp_min(1e-8)
    return soft, hard

def gold_supervised_loss(logits, marker_mask, selected):
    masked = logits.float().masked_fill(~marker_mask, -1e4)
    gold_soft, gold_hard = _gold_target_tensors(selected, marker_mask, logits.device)
    logp = torch.log_softmax(masked, dim=-1)
    p = torch.softmax(masked, dim=-1)

    # Full probability-target training.
    soft_ce = -(gold_soft * logp).sum(-1).mean()

    # Explicit official hard-label target.
    hard_ce = F.cross_entropy(masked, gold_hard)

    # Calibration pressure toward the complete benchmark distribution.
    brier = (((p - gold_soft) ** 2) * marker_mask.float()).sum(-1).mean()

    # Hard-label margin: useful when soft gold has close alternatives.
    target_score = masked.gather(1, gold_hard[:, None]).squeeze(1)
    other = masked.clone()
    other.scatter_(1, gold_hard[:, None], -1e4)
    strongest_other = other.max(-1).values
    rank = F.relu(0.10 + strongest_other - target_score).mean()

    total = soft_ce + 0.35 * hard_ce + 0.20 * brier + 0.10 * rank
    return total, {
        "soft_ce": float(soft_ce.detach()),
        "hard_ce": float(hard_ce.detach()),
        "brier": float(brier.detach()),
        "rank": float(rank.detach()),
    }

@torch.no_grad()
def gold_dev_metrics(model, items=dev_items):
    model.eval()
    correct = 0
    total = 0
    kl_values = []
    brier_values = []
    soft_acc_values = []

    for off in range(0, len(items), BATCH):
        selected = items[off:off+BATCH]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits, _ = model(**b)
        require_finite(logits, "gold validation logits")
        mm = b["marker_mask"].bool()
        masked = logits.float().masked_fill(~mm, -1e9)
        p = torch.softmax(masked, dim=-1)
        g, hard = _gold_target_tensors(selected, mm, device)

        pred = p.argmax(-1)
        correct += int((pred == hard).sum())
        total += int(pred.numel())

        kl = (g * (g.clamp_min(1e-8).log() - p.clamp_min(1e-8).log())).sum(-1)
        br = (((p - g) ** 2) * mm.float()).sum(-1)
        soft_acc = g.gather(1, pred[:, None]).squeeze(1)
        kl_values.extend(kl.cpu().tolist())
        brier_values.extend(br.cpu().tolist())
        soft_acc_values.extend(soft_acc.cpu().tolist())

    return {
        "accuracy": correct / max(1, total),
        "kl_from_gold": float(np.mean(kl_values)),
        "brier": float(np.mean(brier_values)),
        "soft_accuracy": float(np.mean(soft_acc_values)),
    }

def direct_gold_train_full(model):
    # Freeze everything first. Train only CeNN cores + decision path.
    model.requires_grad_(False)
    core = v25_core_parameters(model)
    assert len(core) > 0
    for p in core:
        p.requires_grad = True

    decision_params = []
    # No direct target exists for act_head, so leave it frozen.
    for module in (model.head, model.type_emb, model.scorer):
        if module is not None:
            for p in module.parameters():
                p.requires_grad = True
                decision_params.append(p)

    def gold_step(step):
        start = (step*BATCH + 701) % len(train_items)
        selected = [train_items[(start+i) % len(train_items)] for i in range(BATCH)]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits, _ = model(**b)
            require_finite(logits, "full gold training logits")
            return gold_supervised_loss(logits, b["marker_mask"].bool(), selected)[0]
    metrics = recover(
        "full-gold", model, [(core, 1.0), (decision_params, 0.5)],
        [(DIRECT_FULL_CORE_LR, DIRECT_FULL_CORE_LR_MIN), (DIRECT_FULL_HEAD_LR, DIRECT_FULL_HEAD_LR_MIN)],
        DIRECT_FULL_STEPS, gold_step, lambda: gold_dev_metrics(model), gold_checkpoint_key,
    )
    model.eval().requires_grad_(False)
    return metrics

v25_full_gold_dev = direct_gold_train_full(student_full)
# Refresh teacher-agreement diagnostics after the direct supervised phase.
v25_full_quality = validate_pair(student_full, items=dev_items, limit=len(dev_items))
print("V2.5 full-head direct-gold dev:", v25_full_gold_dev)
print("V2.5 full-head post-gold teacher agreement:", v25_full_quality)
torch.cuda.empty_cache()


full-gold step=0001/900 loss=1.3107 lr=[0.00015000000000000001, 2.5e-06] dev={'accuracy': 0.468, 'kl_from_gold': 0.41859365620836614, 'brier': 0.23813972327002558, 'soft_accuracy': 0.4293766631167382}
full-gold step=0050/900 loss=1.7244 lr=[0.0029921110991098466, 4.989266801509996e-05] dev={'accuracy': 0.33, 'kl_from_gold': 0.46673781280592086, 'brier': 0.25266467360476236, 'soft_accuracy': 0.32314835279434917}
full-gold step=0100/900 loss=1.4160 lr=[0.0029417928446101175, 4.920806591306282e-05] dev={'accuracy': 0.348, 'kl_from_gold': 0.46677920043468474, 'brier': 0.2526829903075704, 'soft_accuracy': 0.3592633261284791}
full-gold step=0150/900 loss=2.0362 lr=[0.0028465093497245433, 4.7911691832986986e-05] dev={'accuracy': 0.328, 'kl_from_gold': 0.46630725914984944, 'brier': 0.2524580686627887, 'soft_accuracy': 0.33573332825163377}
full-gold step=0200/900 loss=2.1996 lr=[0.002709295369352675, 4.604483495717926e-05] dev={'accuracy': 0.338, 'kl_from_gold': 0.4945878290422261, 'brier': 0.2

In [11]:
#@title 8. Benchmark helper — latency + CUDA activation peak
def _sync():
    torch.cuda.synchronize()

@torch.inference_mode()
def _forward(model, b):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        return model(**b)

def benchmark_model(model, probe, warmup=3, repeats=10):
    for _ in range(warmup):
        _forward(model, probe)
    _sync()
    start_alloc = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    samples = []
    for _ in range(repeats):
        _sync()
        t0 = time.perf_counter()
        _forward(model, probe)
        _sync()
        samples.append((time.perf_counter()-t0)*1000.0)
    peak = torch.cuda.max_memory_allocated()
    return {
        "median_ms": float(np.median(samples)),
        "p95_ms": float(np.percentile(samples, 95)),
        "activation_peak_delta_mb": float(max(0, peak-start_alloc) / 2**20),
        "total_parameters": int(sum(p.numel() for p in model.parameters())),
    }

probe = batch(dev_items, 0, min(BATCH, 3))
teacher_perf = benchmark_model(teacher, probe)
v25_full_perf = benchmark_model(student_full, probe)
v25_full_report = {
    "quality": v25_full_quality,
    "teacher": teacher_perf,
    "student": v25_full_perf,
    "speedup_vs_teacher": teacher_perf["median_ms"] / max(v25_full_perf["median_ms"], 1e-9),
    "sequence_length": int(probe["input_ids"].shape[1]),
    "batch_size": int(probe["input_ids"].shape[0]),
    "new_cenn_core_parameters": int(sum(p.numel() for p in v25_core_parameters(student_full))),
}
print(json.dumps(v25_full_report, indent=2))

# All models are timed on these same length-stratified DEV examples, without
# selecting the fastest test batch. Report median/p95 across all timed forwards.
ordered_dev = sorted(dev_items, key=lambda item: len(item["ids"]))
perf_probes = []
for fraction in (0.0, 0.25, 0.5, 0.75, 1.0):
    start = min(len(ordered_dev)-BATCH, int(fraction * (len(ordered_dev)-BATCH)))
    probe_batch = collate_items(ordered_dev[start:start+BATCH], int(tokenizer.pad_token_id))
    perf_probes.append({k: v.to(device) for k, v in probe_batch.items()})

@torch.inference_mode()
def benchmark_suite(model, probes=perf_probes, repeats=20):
    timings, peaks = [], []
    for pb in probes:
        for _ in range(5):
            _forward(model, pb)
        _sync()
        base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
        for _ in range(repeats):
            _sync()
            started = time.perf_counter()
            _forward(model, pb)
            _sync()
            timings.append((time.perf_counter()-started)*1000)
        peaks.append(max(0, torch.cuda.max_memory_allocated()-base) / 2**20)
    return {
        "median_ms": float(np.median(timings)), "p95_ms": float(np.percentile(timings, 95)),
        "activation_peak_delta_mb": float(max(peaks)),
        "total_parameters": sum(p.numel() for p in model.parameters()),
        "probe_lengths": [int(pb["input_ids"].shape[1]) for pb in probes],
        "batch_size": BATCH, "repeats_per_probe": repeats,
        "scope": "model forward only, length-stratified dev probes; excludes tokenization",
    }


{
  "quality": {
    "agreement": 0.516,
    "mean_js": 0.08075470844854135
  },
  "teacher": {
    "median_ms": 142.60589300010906,
    "p95_ms": 147.49055535030493,
    "activation_peak_delta_mb": 55.2626953125,
    "total_parameters": 421293827
  },
  "student": {
    "median_ms": 119.58223599958728,
    "p95_ms": 121.27308620001713,
    "activation_peak_delta_mb": 47.9892578125,
    "total_parameters": 324870461
  },
  "speedup_vs_teacher": 1.192534090101821,
  "sequence_length": 486,
  "batch_size": 3,
  "new_cenn_core_parameters": 21017146
}


In [12]:
#@title 9. Compact pooled-context MLP head — same three training phases
def train_compact_model():
    compact = PooledMarkerDecisionModel(
        student_full, compact_dim=COMPACT_DIM, layers=COMPACT_LAYERS
    ).to(device)
    compact.encoder.requires_grad_(False)
    compact.type_emb.requires_grad_(False)
    head = [p for name,p in compact.named_parameters()
            if not name.startswith(("encoder.", "type_emb."))]
    for p in head: p.requires_grad_(True)

    def distillation_step(step, offset):
        b = batch(train_items, step*BATCH + offset)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta = teacher(**b)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa = compact(**b)
            return distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)
    evaluate = lambda: validate_pair(compact, items=dev_items, limit=len(dev_items))

    # Phase 1: head distillation with the encoder fixed.
    recover("compact-head", compact, [(head, 1.0)], [(COMPACT_HEAD_LR, COMPACT_HEAD_LR_MIN)],
            COMPACT_STEPS, lambda step: distillation_step(step, 101), evaluate, agreement_score)

    # Phase 2: joint CeNN + compact head distillation.
    core = v25_core_parameters(compact)
    assert core, "No CeNN parameters found for joint recovery"
    for p in core: p.requires_grad_(True)
    recover("compact-joint", compact, [(core, 1.0), (head, 0.5)],
            [(JOINT_CORE_LR, JOINT_CORE_LR_MIN), (JOINT_HEAD_LR, JOINT_HEAD_LR_MIN)],
            COMPACT_JOINT_STEPS, lambda step: distillation_step(step, 401), evaluate, agreement_score)

    # Phase 3: direct gold supervision. The action head has no gold target.
    compact.requires_grad_(False)
    for p in core: p.requires_grad_(True)
    direct_head = []
    for module in (compact.type_emb, compact.in_proj, compact.context_proj, compact.marker_mixer, compact.scorer):
        for p in module.parameters():
            p.requires_grad_(True)
            direct_head.append(p)
    def gold_step(step):
        start = (step*BATCH + 1301) % len(train_items)
        selected = [train_items[(start+i) % len(train_items)] for i in range(BATCH)]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits, _ = compact(**b)
            require_finite(logits, "compact gold training logits")
            return gold_supervised_loss(logits, b["marker_mask"].bool(), selected)[0]
    metrics = recover(
        "compact-gold", compact, [(core, 1.0), (direct_head, 0.5)],
        [(DIRECT_COMPACT_CORE_LR, DIRECT_COMPACT_CORE_LR_MIN), (DIRECT_COMPACT_HEAD_LR, DIRECT_COMPACT_HEAD_LR_MIN)],
        DIRECT_COMPACT_STEPS, gold_step, lambda: gold_dev_metrics(compact), gold_checkpoint_key,
    )
    compact.eval().requires_grad_(False)
    print("V2.5 compact direct-gold dev:", metrics)
    return compact

compact_student = train_compact_model() if RUN_COMPACT_HEAD else None
if compact_student is not None:
    assert attention_audit(compact_student)["delta_free"]


compact-head step=0001/700 loss=1.6797 lr=[5e-05] dev={'agreement': 0.364, 'mean_js': 0.11238316984966513}
compact-head step=0050/700 loss=0.6356 lr=[0.0009964047050623564] dev={'agreement': 0.534, 'mean_js': 0.08724893704126589}
compact-head step=0100/700 loss=0.5393 lr=[0.000973575684451737] dev={'agreement': 0.54, 'mean_js': 0.07980869036348304}
compact-head step=0150/700 loss=0.9385 lr=[0.0009308426319301475] dev={'agreement': 0.548, 'mean_js': 0.07952199419034878}
compact-head step=0200/700 loss=0.9377 lr=[0.0008704823553211247] dev={'agreement': 0.534, 'mean_js': 0.07757306177147984}
compact-head step=0250/700 loss=0.7555 lr=[0.0007957108371465079] dev={'agreement': 0.536, 'mean_js': 0.08195559354918078}
compact-head step=0300/700 loss=0.7377 lr=[0.0007105118877711594] dev={'agreement': 0.514, 'mean_js': 0.08539839581120759}
compact-head step=0350/700 loss=0.7537 lr=[0.0006194248888457794] dev={'agreement': 0.536, 'mean_js': 0.07602747331996215}
compact-head step=0400/700 loss=0.

In [13]:
#@title 10. REAL held-out test — teacher vs V2.5 full vs V2.5 compact
# The official LocalLLaMA/typed-decisions test split is loaded and used here for the first time.
# It contains 400 cases / 2,000 decisions. Checkpoint selection above used dev only.

test_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="test"))[:TEST_CASES]
test_items = make_items_from_rows(test_rows, SEED + 2, shuffle_items=False)
assert len(test_rows) == TEST_CASES
assert len(test_items) == 5 * len(test_rows), (
    "Official typed-decisions test should contain exactly five decisions per case."
)
print("official test loaded:", len(test_rows), "cases /", len(test_items), "decisions")

@torch.inference_mode()
def collect_test_records(model, items):
    model.eval()
    records = []
    for off in range(0, len(items), BATCH):
        selected = items[off:off+BATCH]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}
        logits, act_logits = _forward(model, b)
        require_finite(logits, "test logits")
        require_finite(act_logits, "test action logits")
        mm = b["marker_mask"].bool()
        probs = torch.softmax(logits.float().masked_fill(~mm, -1e9), dim=-1)

        for j, item in enumerate(selected):
            k = len(item["gold_probs"])
            p = probs[j, :k].detach().cpu().numpy().astype(np.float64)
            p = np.clip(p, 1e-12, None)
            p /= p.sum()
            records.append({
                "id": f"{item['case_id']}/{item['qid']}",
                "case_id": item["case_id"],
                "qid": item["qid"],
                "workflow": item["workflow"],
                "qtype": item["qtype_name"],
                "labels": item["labels"],
                "p": p.tolist(),
                "g": list(map(float, item["gold_probs"])),
                "gold_index": int(item["gold_index"]),
                "pred_index": int(p.argmax()),
                "action_pred": int(act_logits[j].float().argmax().item()),
            })
    return records

def _ece_top_label(records, bins=10):
    total = len(records)
    score = 0.0
    for b in range(bins):
        bucket = []
        for r in records:
            conf = max(r["p"])
            bi = min(bins - 1, int(conf * bins))
            if bi == b:
                bucket.append((conf, r["pred_index"] == r["gold_index"]))
        if bucket:
            avg_conf = float(np.mean([x[0] for x in bucket]))
            avg_acc = float(np.mean([x[1] for x in bucket]))
            score += len(bucket) / total * abs(avg_conf - avg_acc)
    return float(score)

def typed_gold_metrics(records):
    eps = 1e-12
    correct = []
    soft_acc = []
    kl = []
    tv = []
    brier = []
    score_mae = []
    within_one = []

    for r in records:
        p = np.asarray(r["p"], dtype=np.float64)
        g = np.asarray(r["g"], dtype=np.float64)
        pred = int(r["pred_index"])
        gold = int(r["gold_index"])
        correct.append(pred == gold)
        soft_acc.append(float(g[pred]))
        kl.append(float(np.sum(g * np.log(np.clip(g, eps, None) / np.clip(p, eps, None)))))
        tv.append(float(0.5 * np.abs(p - g).sum()))
        brier.append(float(np.square(p - g).sum()))
        if r["qtype"] == "score":
            idx = np.arange(len(p), dtype=np.float64)
            gap = abs(float((idx * p).sum() - (idx * g).sum()))
            score_mae.append(gap)
            within_one.append(gap <= 1.0)

    by_type = {}
    for kind in ("choice", "score", "noul"):
        sub = [r for r in records if r["qtype"] == kind]
        if sub:
            by_type[kind] = float(np.mean([
                r["pred_index"] == r["gold_index"] for r in sub
            ]))

    by_workflow = {}
    for wf in sorted({r["workflow"] for r in records}):
        sub = [r for r in records if r["workflow"] == wf]
        by_workflow[wf] = float(np.mean([
            r["pred_index"] == r["gold_index"] for r in sub
        ]))

    return {
        "decisions": int(len(records)),
        "cases": int(len({r["case_id"] for r in records})),
        "accuracy": float(np.mean(correct)),
        "soft_accuracy": float(np.mean(soft_acc)),
        "kl_from_gold": float(np.mean(kl)),
        "total_variation": float(np.mean(tv)),
        "brier": float(np.mean(brier)),
        "ece_10bin": _ece_top_label(records, bins=10),
        "score_mae": float(np.mean(score_mae)) if score_mae else None,
        "score_within_one": float(np.mean(within_one)) if within_one else None,
        "accuracy_by_type": by_type,
        "accuracy_by_workflow": by_workflow,
    }

def paired_vs_teacher(teacher_records, student_records, draws=3000):
    assert len(teacher_records) == len(student_records)
    t_by_id = {r["id"]: r for r in teacher_records}
    s_by_id = {r["id"]: r for r in student_records}
    assert set(t_by_id) == set(s_by_id)

    ids = sorted(t_by_id)
    t_ok = np.asarray([
        t_by_id[i]["pred_index"] == t_by_id[i]["gold_index"] for i in ids
    ], dtype=np.float64)
    s_ok = np.asarray([
        s_by_id[i]["pred_index"] == s_by_id[i]["gold_index"] for i in ids
    ], dtype=np.float64)

    student_only = int(np.sum((s_ok == 1) & (t_ok == 0)))
    teacher_only = int(np.sum((s_ok == 0) & (t_ok == 1)))
    both_correct = int(np.sum((s_ok == 1) & (t_ok == 1)))
    both_wrong = int(np.sum((s_ok == 0) & (t_ok == 0)))

    agreements, top2_hits, action_agree, js = [], [], [], []
    for i in ids:
        tr, sr = t_by_id[i], s_by_id[i]
        tp = np.asarray(tr["p"], dtype=np.float64)
        sp = np.asarray(sr["p"], dtype=np.float64)
        agreements.append(tr["pred_index"] == sr["pred_index"])
        top2_hits.append(tr["pred_index"] in np.argsort(sp)[-min(2, len(sp)):])
        action_agree.append(tr["action_pred"] == sr["action_pred"])
        mid = 0.5 * (tp + sp)
        j = 0.5 * np.sum(tp * np.log(np.clip(tp, 1e-12, None) / np.clip(mid, 1e-12, None)))
        j += 0.5 * np.sum(sp * np.log(np.clip(sp, 1e-12, None) / np.clip(mid, 1e-12, None)))
        js.append(float(j))

    # Paired bootstrap over CASES (not individual decisions), preserving each case's 5 questions.
    case_to_gaps = {}
    for idx, i in enumerate(ids):
        case_to_gaps.setdefault(t_by_id[i]["case_id"], []).append(float(s_ok[idx] - t_ok[idx]))
    case_gaps = np.asarray([np.mean(v) for v in case_to_gaps.values()], dtype=np.float64)
    rng = np.random.default_rng(SEED + 91)
    boot = np.empty(draws, dtype=np.float64)
    for d in range(draws):
        boot[d] = rng.choice(case_gaps, size=len(case_gaps), replace=True).mean()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    gap = float(s_ok.mean() - t_ok.mean())

    if lo > 0:
        verdict = "student_better"
    elif hi < 0:
        verdict = "teacher_better"
    else:
        verdict = "no_clear_difference"

    return {
        "accuracy_gap_student_minus_teacher": gap,
        "accuracy_gap_95pct_case_bootstrap": [float(lo), float(hi)],
        "paired_verdict": verdict,
        "student_only_correct": student_only,
        "teacher_only_correct": teacher_only,
        "both_correct": both_correct,
        "both_wrong": both_wrong,
        "decision_agreement": float(np.mean(agreements)),
        "teacher_top1_in_student_top2": float(np.mean(top2_hits)),
        "action_head_agreement": float(np.mean(action_agree)),
        "mean_js_divergence": float(np.mean(js)),
        "p95_js_divergence": float(np.percentile(js, 95)),
    }

models = {"teacher": teacher, "v25_full_head": student_full}
if compact_student is not None:
    models["v25_compact"] = compact_student

print("\nRunning untouched official test: 400 cases / 2,000 decisions ...")
test_records = {}
for name, model in models.items():
    print("  evaluating", name)
    test_records[name] = collect_test_records(model, test_items)

# Latency on the same held-out test examples (small fixed probe for repeatability).
test_probe_items = test_items[:min(BATCH, len(test_items))]
test_probe = collate_items(test_probe_items, int(tokenizer.pad_token_id))
test_probe = {k:v.to(device) for k,v in test_probe.items()}

report = {
    "architecture": "v25_sliding_cenn_multiscale_cenn",
    "source_model": SOURCE_MODEL,
    "training_revision": "2026-10-06-cenn-recovery-rollback",
    "recovery_settings": {
        "schedule": "local_cosine_unchanged_joint_warmup_cosine_with_backoff",
        "warmup_steps": RECOVERY_WARMUP,
        "max_restarts": RECOVERY_MAX_RESTARTS,
        "backoff": RECOVERY_BACKOFF,
        "local_lr": [LOCAL_LR, LOCAL_LR_MIN],
        "full_gold_core_lr": [DIRECT_FULL_CORE_LR, DIRECT_FULL_CORE_LR_MIN],
        "full_gold_head_lr": [DIRECT_FULL_HEAD_LR, DIRECT_FULL_HEAD_LR_MIN],
        "compact_gold_core_lr": [DIRECT_COMPACT_CORE_LR, DIRECT_COMPACT_CORE_LR_MIN],
        "compact_gold_head_lr": [DIRECT_COMPACT_HEAD_LR, DIRECT_COMPACT_HEAD_LR_MIN],
        "compact_head_lr": [COMPACT_HEAD_LR, COMPACT_HEAD_LR_MIN],
        "joint_core_lr": [JOINT_CORE_LR, JOINT_CORE_LR_MIN],
        "joint_head_lr": [JOINT_HEAD_LR, JOINT_HEAD_LR_MIN],
        "global_core_lr": [GLOBAL_CORE_LR, GLOBAL_CORE_LR_MIN],
        "global_head_lr": [GLOBAL_HEAD_LR, GLOBAL_HEAD_LR_MIN],
        "marker_alignment_weight": MARKER_ALIGNMENT_WEIGHT,
        "gold_checkpoint_selection": "accuracy_then_kl_then_brier_including_stage_baseline",
        "amp_initial_scale": 128.0,
    },
    "test_protocol": {
        "dataset": "LocalLLaMA/typed-decisions",
        "config": "all",
        "split": "test",
        "cases": len(test_rows),
        "decisions": len(test_items),
        "used_for_training": False,
        "used_for_checkpoint_selection": False,
        "dev_source": "held-out subset of official train split",
    },
    "full_attention_layers_replaced": full_layers,
    "all_encoder_layers_replaced": replacement_layers,
    "architecture_config": ARCHITECTURE,
    "source_revision": source_dir.name,
    "runtime": {"torch": torch.__version__, "dtype": str(amp_dtype),
                "gpu": torch.cuda.get_device_name(), "transformers": __import__("transformers").__version__},
    "recovery_phases": recovery_reports,
    "models": {},
}

teacher_gold = typed_gold_metrics(test_records["teacher"])
teacher_perf = benchmark_suite(teacher)
report["models"]["teacher"] = {
    "gold_test": teacher_gold,
    "performance": teacher_perf,
    "quality_vs_teacher": {
        "decision_agreement": 1.0,
        "mean_js_divergence": 0.0,
    },
}

for name, model in models.items():
    if name == "teacher":
        continue
    gold_metrics = typed_gold_metrics(test_records[name])
    perf = benchmark_suite(model)
    paired = paired_vs_teacher(test_records["teacher"], test_records[name])
    report["models"][name] = {
        "gold_test": gold_metrics,
        "performance": perf,
        "speedup_vs_teacher": teacher_perf["median_ms"] / max(perf["median_ms"], 1e-9),
        "parameter_ratio_vs_teacher": perf["total_parameters"] / teacher_perf["total_parameters"],
        "quality_vs_teacher": paired,
    }

# Primary quality ranking: official hard-label accuracy, then KL and Brier as tie-breakers.
ranking = sorted(
    report["models"],
    key=lambda name: (
        -report["models"][name]["gold_test"]["accuracy"],
        report["models"][name]["gold_test"]["kl_from_gold"],
        report["models"][name]["gold_test"]["brier"],
    ),
)
report["real_test_quality_winner"] = ranking[0]
report["real_test_quality_ranking"] = ranking
report["fastest_model"] = min(
    report["models"],
    key=lambda name: report["models"][name]["performance"]["median_ms"],
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / "v25_sliding_cenn_REAL_TEST_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n=== REAL HELD-OUT GOLD TEST ===")
print("model               accuracy    KL       Brier    ECE      latency    speed")
print("-" * 82)
for name, result in report["models"].items():
    g = result["gold_test"]
    p = result["performance"]
    speed = 1.0 if name == "teacher" else result["speedup_vs_teacher"]
    print(
        f"{name:18s}  {g['accuracy']:.3%}   {g['kl_from_gold']:.4f}   "
        f"{g['brier']:.4f}   {g['ece_10bin']:.4f}   "
        f"{p['median_ms']:.2f} ms   {speed:.2f}x"
    )
    print("  by type:", {k: f"{v:.2%}" for k, v in g["accuracy_by_type"].items()})

print("\nQUALITY WINNER:", report["real_test_quality_winner"])
print("FASTEST MODEL :", report["fastest_model"])

for name in report["models"]:
    if name == "teacher":
        continue
    q = report["models"][name]["quality_vs_teacher"]
    lo, hi = q["accuracy_gap_95pct_case_bootstrap"]
    print(f"\n{name} vs teacher")
    print(
        f"  accuracy gap : {q['accuracy_gap_student_minus_teacher']:+.2%} "
        f"(95% case-bootstrap CI {lo:+.2%} to {hi:+.2%})"
    )
    print("  verdict      :", q["paired_verdict"])
    print(
        "  student-only / teacher-only correct:",
        q["student_only_correct"], "/", q["teacher_only_correct"],
    )
    print(f"  agreement    : {q['decision_agreement']:.2%}")
    print(f"  top-2 recall : {q['teacher_top1_in_student_top2']:.2%}")
    print(f"  mean JS      : {q['mean_js_divergence']:.5f}")

print("\nsaved:", report_path)


official test loaded: 400 cases / 2000 decisions

Running untouched official test: 400 cases / 2,000 decisions ...
  evaluating teacher
  evaluating v25_full_head
  evaluating v25_compact

=== REAL HELD-OUT GOLD TEST ===
model               accuracy    KL       Brier    ECE      latency    speed
----------------------------------------------------------------------------------
teacher             76.550%   0.1023   0.0552   0.1273   90.86 ms   1.00x
  by type: {'choice': '73.33%', 'score': '72.12%', 'noul': '85.67%'}
v25_full_head       50.950%   0.3498   0.1910   0.0282   83.41 ms   1.09x
  by type: {'choice': '42.17%', 'score': '45.12%', 'noul': '67.50%'}
v25_compact         52.500%   0.3116   0.1698   0.0703   68.87 ms   1.32x
  by type: {'choice': '43.00%', 'score': '44.50%', 'noul': '72.67%'}

QUALITY WINNER: teacher
FASTEST MODEL : v25_compact

v25_full_head vs teacher
  accuracy gap : -25.60% (95% case-bootstrap CI -28.60% to -22.70%)
  verdict      : teacher_better
  student-on

## Interpretation

The teacher is the unchanged source checkpoint. `v25_full_head` is an intermediate retaining the original Transformer decision head; only `v25_compact`, with the pooled MLP head and all-CeNN encoder, is eligible for export.

Gold accuracy and paired case-bootstrap intervals determine fidelity; teacher agreement is diagnostic. Top-2 recall includes binary questions and is not a standalone quality criterion. Sliding and global attention are replaced by cellular recurrent layers, while pretrained FFNs, norms and residuals remain.

Local transfer retains 0.01 to 0.002. Joint and gold phases use lower rates with warmup, fresh AMP scaling and bounded checkpoint rollback after the observed Stage B failure; finite-update safeguards, gradient clipping and best-dev checkpoint selection remain enabled. Stage A can resume from the new architecture-specific output directory. Keep it on persistent storage to survive runtime resets. This notebook has no inherited accuracy or latency measurements; run the GPU evaluation before judging quality or speed.


In [14]:
#@title 11. Save, reload and upload ONLY if the final model passes
import os, gc, sys
from huggingface_hub import HfApi, get_token, notebook_login

if compact_student is None:
    print("No final pooled-head model: export skipped. Enable RUN_COMPACT_HEAD and run again.")
else:
    audit = attention_audit(compact_student)
    assert audit["delta_free"], "Sliding CeNN export must contain no Delta layers"
    gate = export_gate(
        report["models"]["teacher"], report["models"]["v25_compact"], audit,
        max_accuracy_drop=MAX_ACCURACY_DROP, min_speedup=MIN_SPEEDUP,
    )
    report["export_gate"] = gate
    report["attention_audit"] = audit
    report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
    print(json.dumps(gate, indent=2))
    if not gate["passed"]:
        print("Model did not pass: report saved, Hugging Face upload skipped.")
    else:
        # Save enough information for exact reconstruction without teacher downloads.
        export_dir = OUTPUT_DIR / "accepted_model"
        save_v25(compact_student, export_dir, tokenizer, ARCHITECTURE, source_cfg, report)
        verification_batch = perf_probes[2]
        compact_student.to(device)
        expected = tuple(x.detach().float().cpu() for x in _forward(compact_student, verification_batch))
        # Offload existing models before loading a second copy on the GPU.
        for existing in models.values():
            existing.to("cpu")
        gc.collect(); torch.cuda.empty_cache()
        reloaded = load_v25(export_dir, device="cuda", backend=None)
        actual = tuple(x.detach().float().cpu() for x in _forward(reloaded, verification_batch))
        for a, e in zip(actual, expected):
            torch.testing.assert_close(a, e, rtol=0.01, atol=0.01)
        assert attention_audit(reloaded)["passed"]
        report["export_reload_verified"] = True
        (export_dir / "evaluation.json").write_text(json.dumps(report, indent=2, allow_nan=False))
        report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
        del reloaded
        gc.collect(); torch.cuda.empty_cache()
        result = report["models"]["v25_compact"]
        card = f"""---
base_model: {SOURCE_MODEL}
tags:
- tinycenn
- recurrent-memory
- typed-decisions
- custom-code
---
# Laya V2.5: Sliding CeNN + Multiscale CeNN

Bidirectional typed-decision encoder. Sliding attention is replaced with bidirectional sliding CeNN;
global attention with multiscale recurrent CeNN; the final head is a pooled-context option MLP.
Encoder FFNs, normalization and residual connections are retained. This is not a causal text generator.

Source checkpoint: {SOURCE_MODEL}, revision `{source_dir.name}`. Follow the source model's license
and usage conditions; this export does not grant a new license to inherited weights.

## Measured evaluation
- Gold test accuracy: {result['gold_test']['accuracy']:.4%}
- Speedup vs unchanged teacher: {result['speedup_vs_teacher']:.3f}x
- Test cases: {result['gold_test']['cases']}
- GPU: {torch.cuda.get_device_name()}
- Dtype: {amp_dtype}
- Full metrics, CI, architecture and acceptance thresholds: `evaluation.json`
- Save/reload equivalence: passed.

Latency covers model forward only on length-stratified dev batches, excluding tokenization.
The 400-case benchmark was used in prior research iterations; this is not independent publication validation.
Action-head outputs are distilled from the teacher, not validated against gold action labels.

## Load
Install `requirements.txt` (PyTorch; no FLA dependency). The included `tinycenn_lm` folder contains the runtime;
run Python from the downloaded model directory (or add that directory to sys.path).
Use the tokenizer in `tokenizer/` and the typed-decision packing utilities in
`tinycenn_lm.standalone_decision`.

```python
from tinycenn_lm.decision_v25 import load_v25
model = load_v25(".", device="cuda", backend=None)
# CPU inference: load_v25(".", device="cpu", backend=None)
```
CUDA inference should run under torch.autocast with native BF16 or FP16 as used in evaluation.
CPU and CUDA use the same parallel convolutional recurrence. GPU speed is measured separately.
"""
        (export_dir / "README.md").write_text(card)
        print("Accepted checkpoint saved and reload verified:", export_dir)
        if PUSH_TO_HUB:
            token = get_token()
            if not token:
                try:
                    from google.colab import userdata
                    token = userdata.get("HF_TOKEN")
                except Exception:
                    token = None
            if not token:
                notebook_login()
                token = get_token()
            if not token:
                raise RuntimeError("Accepted model saved locally. Authenticate and rerun this cell to upload.")
            api = HfApi(token=token)
            repo_id = HF_REPO_ID.strip() or f"{api.whoami()['name']}/Laya-V25-SlidingCeNN"
            api.create_repo(repo_id, repo_type="model", private=HF_PRIVATE, exist_ok=True)
            commit = api.upload_folder(
                repo_id=repo_id, repo_type="model", folder_path=str(export_dir),
                commit_message="Upload evaluated V2.5 sliding-CeNN decision model",
                ignore_patterns=["**/__pycache__/**", "**/*.pyc"],
            )
            print("Hugging Face model:", f"https://huggingface.co/{repo_id}")
            print("Uploaded commit:", commit.oid)


{
  "passed": false,
  "reasons": [
    "accuracy loss exceeds tolerance",
    "paired confidence interval exceeds tolerance"
  ],
  "max_accuracy_drop": 0.02,
  "min_speedup": 1.1
}
Model did not pass: report saved, Hugging Face upload skipped.
